# 다중시점 historical CLV M2 — 체크포인트 진단

기존 seed 42 역사적 개발실험의 M1/M2 체크포인트만 다시 읽습니다. 모델을 재학습하거나 checkpoint를 선택하지 않습니다.

1. M1↔M2 전체 변화, M2 공동학습 경로 변화, CLV 직접 효과의 Top-K 목록 변경률
2. 전체 후보상품에서 각 점수 변화의 표준편차와 M1 점수 대비 비율
3. M2 내부 `rho=0` 순위 경계와 CLV 직접 점수 변화 비교
4. 네 학습시점과 최종 평가시점 사이 historical CLV 조건값 변화


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'a284c74c25016b54bd3971d93ba6bfb41b369ae3'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('진단 코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_dynamic_multianchor_diagnostic import (
    configure_dynamic_multianchor_diagnostic,
    preflight_summary,
    run_dynamic_multianchor_diagnostic,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_dynamic_multianchor_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_dynamic_clv_multianchor_historical_screen_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
report = run_dynamic_multianchor_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) Top-K 추천목록 변경률')
display(report['topk_change'])
print('2) 후보점수 실효 크기')
display(report['score_effect'])
print('3) 순위 경계 대비 CLV 직접 영향')
display(report['rank_boundary'])
print('4) 시점별 historical CLV 변화')
display(report['condition_variation'])
print('결과 파일:', report['paths'])